<a href="https://colab.research.google.com/github/salphonseds/llm-from-scratch/blob/main/notebooks/09_Full_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Notebook 09: Full Fine-tuning
# Cell 1 — Load pretrained GPT-2 small and map it onto our architecture

import torch, torch.nn as nn, torch.nn.functional as F
import math, time

from transformers import GPT2LMHeadModel, GPT2TokenizerFast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)   # "gpt2" = GPT-2 small, 124M
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

cfg = model.config
print(f"vocab_size : {cfg.vocab_size}      (ours: 50257 — same tiktoken GPT-2 BPE)")
print(f"d_model    : {cfg.n_embd}        (ours: 512)")
print(f"num_layers : {cfg.n_layer}         (ours: 6)")
print(f"num_heads  : {cfg.n_head}         (ours: 8)")
print(f"max_len    : {cfg.n_positions}       (ours: 256)")

n_params = model.num_parameters()
print(f"\nparameters : {n_params:,}")

# Weight tying check — the same trick from our notebook 04
tied = model.lm_head.weight.data_ptr() == model.transformer.wte.weight.data_ptr()
print(f"weight tying (lm_head is wte): {tied}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

vocab_size : 50257      (ours: 50257 — same tiktoken GPT-2 BPE)
d_model    : 768        (ours: 512)
num_layers : 12         (ours: 6)
num_heads  : 12         (ours: 8)
max_len    : 1024       (ours: 256)

parameters : 124,439,808
weight tying (lm_head is wte): True


In [ ]:
# Cell 2 — Baseline: perplexity on general text (WikiText-2 test)

from datasets import load_dataset

wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
general_text = "\n\n".join(t for t in wikitext["text"] if t.strip())

general_ids = tokenizer(general_text, return_tensors="pt").input_ids[0]
print(f"WikiText-2 test: {len(general_ids):,} tokens")

@torch.no_grad()
def perplexity(model, ids, ctx_len=1024, max_windows=None):
    """Mean NLL over non-overlapping windows -> exp. Simple + fast; slightly
    pessimistic vs. sliding-window eval (early tokens in each window get
    little context), but the bias is identical before/after fine-tuning,
    which is all we need for a forgetting delta."""
    model.eval()
    nll_sum, tok_count = 0.0, 0
    windows = range(0, len(ids) - 1, ctx_len)
    if max_windows is not None:
        windows = list(windows)[:max_windows]
    for start in windows:
        chunk = ids[start : start + ctx_len + 1].to(device)
        if len(chunk) < 2:
            break
        x, y = chunk[:-1].unsqueeze(0), chunk[1:].unsqueeze(0)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(x).logits
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.view(-1))
        n = y.numel()
        nll_sum += loss.item() * n
        tok_count += n
    return math.exp(nll_sum / tok_count), tok_count

t0 = time.time()
ppl_general_before, n_tok = perplexity(model, general_ids)
print(f"Baseline general-text perplexity: {ppl_general_before:.2f} "
      f"({n_tok:,} tokens, {time.time()-t0:.1f}s)")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (286177 > 1024). Running this sequence through the model will result in indexing errors


WikiText-2 test: 286,177 tokens
Baseline general-text perplexity: 29.03 (286,176 tokens, 11.3s)


In [ ]:
# Cell 3 — Baseline generation snapshot (re-run verbatim after fine-tuning)

PROMPTS = [
    "The scientists analyzed the data and concluded that",   # modern/technical register
    "In today's news, the government announced",             # modern/journalistic register
    "The king stood at the window and",                      # neutral — contestable ground
]

@torch.no_grad()
def sample(model, prompt, max_new_tokens=60, temperature=0.8, top_k=50, seed=42):
    torch.manual_seed(seed)                       # same seed -> controlled A/B
    model.eval()
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    out = model.generate(
        ids,
        max_new_tokens=max_new_tokens,
        do_sample=True, temperature=temperature, top_k=top_k,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

for p in PROMPTS:
    print("=" * 70)
    print(sample(model, p))
print("=" * 70)

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


The scientists analyzed the data and concluded that for the 10 years between 2005 and 2010, a total of 1,068 patients received at least one treatment at a time. That represented a 20 percent decrease in the number of hospitalizations in the United States.

Some of the data may have been skewed by the fact that the number of
In today's news, the government announced for the first time details about the government's plan to close some 5,000 schools across India. The government wants to close at least 50 schools and one kindergarten in each province in each state by the end of 2016.

The government's plan is to close about 50 schools across India by the
The king stood at the window and looked to the king who was looking at him.

"Father, it is not at all time to do something like this, O King. Let us go, and then we will see how far we have come".

King Arthur was greatly excited, and was about to speak, when


In [ ]:
# Cell 4 — Domain corpus: Tiny Shakespeare (download, tokenize, split, baseline PPL)

import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
shakespeare_text = urllib.request.urlopen(url).read().decode("utf-8")
print(f"characters : {len(shakespeare_text):,}")
print("--- first 250 chars ---")
print(shakespeare_text[:250])

shake_ids = tokenizer(shakespeare_text, return_tensors="pt").input_ids[0]
print(f"\nGPT-2 BPE tokens : {len(shake_ids):,}  "
      f"({len(shakespeare_text)/len(shake_ids):.2f} chars/token)")

# 90/10 train/val split — contiguous, not shuffled (it's one long document)
split = int(0.9 * len(shake_ids))
shake_train, shake_val = shake_ids[:split], shake_ids[split:]
print(f"train {len(shake_train):,} tokens | val {len(shake_val):,} tokens")

# Domain baseline: how "surprising" is Shakespeare to stock GPT-2?
ppl_domain_before, n = perplexity(model, shake_val)
print(f"\nBaseline domain (Shakespeare val) perplexity: {ppl_domain_before:.2f}")
print(f"Baseline general (WikiText-2)     perplexity: {ppl_general_before:.2f}")

characters : 1,115,394
--- first 250 chars ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.


GPT-2 BPE tokens : 338,025  (3.30 chars/token)
train 304,222 tokens | val 33,803 tokens

Baseline domain (Shakespeare val) perplexity: 55.16
Baseline general (WikiText-2)     perplexity: 29.03


In [ ]:
# Cell 5 — Fine-tune GPT-2 on Shakespeare (fp16 + GradScaler, warmup + cosine)

BATCH, SEQ_LEN   = 4, 512
STEPS, WARMUP    = 300, 20
LR_PEAK          = 3e-5          # ~10x below pretraining LR — the drift knob

def get_batch(ids, batch=BATCH, seq_len=SEQ_LEN):
    starts = torch.randint(0, len(ids) - seq_len - 1, (batch,))
    x = torch.stack([ids[s : s + seq_len]         for s in starts]).to(device)
    y = torch.stack([ids[s + 1 : s + seq_len + 1] for s in starts]).to(device)
    return x, y

def lr_at(step):
    if step < WARMUP:
        return LR_PEAK * (step + 1) / WARMUP
    p = (step - WARMUP) / (STEPS - WARMUP)
    return 0.1 * LR_PEAK + 0.9 * LR_PEAK * 0.5 * (1 + math.cos(math.pi * p))

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PEAK, weight_decay=0.01)
scaler    = torch.amp.GradScaler("cuda")
torch.manual_seed(42)

model.train()
t0 = time.time()
for step in range(STEPS):
    for g in optimizer.param_groups:
        g["lr"] = lr_at(step)

    x, y = get_batch(shake_train)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(x).logits
    loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.view(-1))

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)                       # unscale BEFORE clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if step % 50 == 0 or step == STEPS - 1:
        ppl_val, _ = perplexity(model, shake_val)
        model.train()                                # perplexity() left it in eval
        print(f"step {step:3d} | lr {lr_at(step):.2e} | train loss {loss.item():.3f} "
              f"| domain val PPL {ppl_val:6.2f} | scale {scaler.get_scale():.0f} "
              f"| {time.time()-t0:5.1f}s")

print(f"\nFine-tuning done in {time.time()-t0:.1f}s")
ppl_domain_after, _ = perplexity(model, shake_val)
print(f"Domain val PPL: {ppl_domain_before:.2f} -> {ppl_domain_after:.2f}")

step   0 | lr 1.50e-06 | train loss 4.368 | domain val PPL  55.16 | scale 32768 |   1.7s
step  50 | lr 2.92e-05 | train loss 3.581 | domain val PPL  29.03 | scale 4096 |  14.8s
step 100 | lr 2.49e-05 | train loss 3.732 | domain val PPL  27.76 | scale 4096 |  28.1s
step 150 | lr 1.80e-05 | train loss 3.279 | domain val PPL  27.37 | scale 4096 |  41.4s
step 200 | lr 1.06e-05 | train loss 3.648 | domain val PPL  27.12 | scale 4096 |  54.9s
step 250 | lr 5.07e-06 | train loss 3.527 | domain val PPL  26.90 | scale 4096 |  68.4s
step 299 | lr 3.00e-06 | train loss 3.329 | domain val PPL  26.81 | scale 4096 |  81.7s

Fine-tuning done in 81.7s
Domain val PPL: 55.16 -> 26.81


In [ ]:
# Cell 6 — The forgetting measurement: general PPL + generation A/B, after fine-tuning

ppl_general_after, _ = perplexity(model, general_ids)

print("PPL                         before      after      change")
print(f"general (WikiText-2)      {ppl_general_before:8.2f}  {ppl_general_after:9.2f}"
      f"   {100*(ppl_general_after/ppl_general_before-1):+6.1f}%")
print(f"domain  (Shakespeare val) {ppl_domain_before:8.2f}  {ppl_domain_after:9.2f}"
      f"   {100*(ppl_domain_after/ppl_domain_before-1):+6.1f}%")

print("\n--- generation snapshot, same prompts, same seed ---")
for p in PROMPTS:
    print("=" * 70)
    print(sample(model, p))
print("=" * 70)

PPL                         before      after      change
general (WikiText-2)         29.03      39.14    +34.8%
domain  (Shakespeare val)    55.16      26.81    -51.4%

--- generation snapshot, same prompts, same seed ---
The scientists analyzed the data and concluded that for the most part, it was due to a natural effect in nature: it was not at all determined to prevent it. A natural effect is caused by a certain kind of matter that may be moved by a natural action.

KING EDWARD IV:
It was said that the earth tremb
In today's news, the government announced for the first time the ban on the use of the word 'theft' in the English tongue
To prevent people from being ignorant of the ancient and ancient law,
That even when the law forbids stealing, it may be enforced
In all cases, and no person shall be arrested
If
The king stood at the window and saw the king come in; and he told me that the king was coming to take supper at night with the king's daughter Atelec.

KING RICHARD II:
'I w

In [ ]:
# Cell 7 — Replay setup: pristine checkpoint + WikiText-2 train tokens + mixed batcher
import gc

# 1) Restore untouched GPT-2 (drop fine-tuned weights if present)
if "model" in globals():
    del model

gc.collect(); torch.cuda.empty_cache()
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

# Sanity: pristine model must reproduce the original baselines
ppl_g, _ = perplexity(model, general_ids)
ppl_d, _ = perplexity(model, shake_val)
print(f"pristine check — general PPL {ppl_g:.2f} (expect 29.03) | "
      f"domain PPL {ppl_d:.2f} (expect 55.16)")

# 2) Replay pool: WikiText-2 *train* split (test split stays untouched — eval only)
wikitext_train = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
general_train_text = "\n\n".join(t for t in wikitext_train["text"] if t.strip())
general_train_ids = tokenizer(general_train_text, return_tensors="pt").input_ids[0]
print(f"replay pool: {len(general_train_ids):,} tokens")

# 3) Mixed batcher: 3 Shakespeare + 1 WikiText per batch (25% replay)
def get_mixed_batch(domain_ids, replay_ids, batch=BATCH, seq_len=SEQ_LEN, n_replay=1):
    xs, ys = [], []
    for i in range(batch):
        src = replay_ids if i < n_replay else domain_ids
        s = torch.randint(0, len(src) - seq_len - 1, (1,)).item()
        xs.append(src[s : s + seq_len])
        ys.append(src[s + 1 : s + seq_len + 1])
    return torch.stack(xs).to(device), torch.stack(ys).to(device)

x, y = get_mixed_batch(shake_train, general_train_ids)
print(f"mixed batch shapes: x {tuple(x.shape)}, y {tuple(y.shape)}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

pristine check — general PPL 29.03 (expect 29.03) | domain PPL 55.16 (expect 55.16)
replay pool: 2,415,650 tokens
mixed batch shapes: x (4, 512), y (4, 512)


In [ ]:
# Cell 8 — Replay fine-tune: identical run, 3 Shakespeare + 1 WikiText per batch

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PEAK, weight_decay=0.01)
scaler    = torch.amp.GradScaler("cuda")
torch.manual_seed(42)

model.train()
t0 = time.time()
for step in range(STEPS):
    for g in optimizer.param_groups:
        g["lr"] = lr_at(step)

    x, y = get_mixed_batch(shake_train, general_train_ids)   # <- the only change
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(x).logits
    loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.view(-1))

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if step % 50 == 0 or step == STEPS - 1:
        ppl_val, _  = perplexity(model, shake_val)
        ppl_gen, _  = perplexity(model, general_ids, max_windows=30)   # quick probe
        model.train()
        print(f"step {step:3d} | lr {lr_at(step):.2e} | train loss {loss.item():.3f} "
              f"| domain PPL {ppl_val:6.2f} | general PPL~ {ppl_gen:6.2f} "
              f"| scale {scaler.get_scale():.0f} | {time.time()-t0:5.1f}s")

print(f"\nReplay fine-tuning done in {time.time()-t0:.1f}s")
ppl_domain_replay, _  = perplexity(model, shake_val)
print(f"Domain val PPL: {ppl_domain_before:.2f} -> {ppl_domain_replay:.2f}")

step   0 | lr 1.50e-06 | train loss 4.265 | domain PPL  55.16 | general PPL~  30.80 | scale 32768 |   2.9s
step  50 | lr 2.92e-05 | train loss 3.663 | domain PPL  29.54 | general PPL~  27.23 | scale 8192 |  17.2s
step 100 | lr 2.49e-05 | train loss 3.790 | domain PPL  28.22 | general PPL~  26.12 | scale 8192 |  31.7s
step 150 | lr 1.80e-05 | train loss 3.283 | domain PPL  27.83 | general PPL~  25.64 | scale 8192 |  46.2s
step 200 | lr 1.06e-05 | train loss 3.689 | domain PPL  27.56 | general PPL~  25.16 | scale 8192 |  60.9s
step 250 | lr 5.07e-06 | train loss 3.778 | domain PPL  27.45 | general PPL~  25.02 | scale 8192 |  75.7s
step 299 | lr 3.00e-06 | train loss 3.353 | domain PPL  27.37 | general PPL~  24.98 | scale 8192 |  90.3s

Replay fine-tuning done in 90.3s
Domain val PPL: 55.16 -> 27.37


In [ ]:
# Cell 9 — Three-way verdict: pristine vs. naive fine-tune vs. replay fine-tune

ppl_general_replay, _ = perplexity(model, general_ids)   # full 286k-token harness

print(f"{'PPL':<28}{'pristine':>10}{'naive FT':>10}{'replay FT':>11}")
print(f"{'general (WikiText-2 test)':<28}{ppl_general_before:>10.2f}"
      f"{ppl_general_after:>10.2f}{ppl_general_replay:>11.2f}")
print(f"{'domain (Shakespeare val)':<28}{ppl_domain_before:>10.2f}"
      f"{ppl_domain_after:>10.2f}{ppl_domain_replay:>11.2f}")

print(f"\nforgetting (general PPL change vs. pristine):")
print(f"  naive : {100*(ppl_general_after /ppl_general_before-1):+6.1f}%")
print(f"  replay: {100*(ppl_general_replay/ppl_general_before-1):+6.1f}%")
print(f"domain gain retained by replay vs. naive: "
      f"{100*(ppl_domain_before-ppl_domain_replay)/(ppl_domain_before-ppl_domain_after):.0f}%")

print("\n--- generation snapshot: replay model, same prompts, same seed ---")
for p in PROMPTS:
    print("=" * 70)
    print(sample(model, p))
print("=" * 70)

PPL                           pristine  naive FT  replay FT
general (WikiText-2 test)        29.03     39.14      23.84
domain (Shakespeare val)         55.16     26.81      27.37

forgetting (general PPL change vs. pristine):
  naive :  +34.8%
  replay:  -17.9%
domain gain retained by replay vs. naive: 98%

--- generation snapshot: replay model, same prompts, same seed ---
The scientists analyzed the data and concluded that for the most part, the "soul of the brain" was located in the anterior part of the brain.

The researchers say the findings are important because they provide new insight into how the brain works.

Some of the data used to make the report was from rats, and the
In today's news, the government announced for the first time the ban on the use of the Internet in Pakistan, which it said would "cancellate all internet service providers and impose heavy fines on any provider who violates this rule".

The decision follows an earlier ban on access to state-owned internet se

In [ ]:
# Cell 10 — Notebook 09 summary

print("="*70)
print("NOTEBOOK 09 COMPLETE: FULL FINE-TUNING  (Days 15-16)")
print("="*70)

print("""
WHAT WE DID
  1. Loaded pretrained GPT-2 small (124,439,808 params) — our notebook-04
     architecture scaled up: same vocab (50257), same weight tying,
     learned pos-emb, pre-LN, GELU. d_model 768 / 12 layers / 12 heads.
  2. Baselines before touching weights: general PPL 29.03 (WikiText-2
     test, 286k tokens) + seeded generation snapshot on 3 fixed prompts.
  3. Domain corpus: Tiny Shakespeare, 338,025 BPE tokens (3.30 chars/tok
     vs ~4 for modern English — distribution shift visible at the
     tokenizer level). Contiguous 90/10 split. Domain baseline PPL 55.16.
  4. Naive fine-tune: 300 steps, batch 4 x seq 512, lr 3e-5 peak
     (warmup+cosine), fp16 + GradScaler, ~82 s on the T4.
  5. Replay fine-tune: identical run from pristine weights, but each
     batch = 3 Shakespeare + 1 WikiText-train sequence (25% replay).

THE 3-WAY VERDICT (full eval harness)
                          pristine   naive FT   replay FT
  general PPL              29.03      39.14      23.84
  domain  PPL              55.16      26.81      27.37
  forgetting                  --      +34.8%     -17.9%
  domain gain retained        --       100%        98%

KEY LESSONS
  * Fine-tuning = continued pretraining on a shifted distribution;
    lr (3e-5, ~10x below pretraining) is the drift-limiting knob.
  * Catastrophic forgetting is real, measurable, and NON-UNIFORM: the
    strongest invaders were the corpus's highest-frequency patterns —
    'NAME:' speaker tags and verse lineation — which colonized even
    modern-register prompts in the naive model.
  * Replay bends the trade-off curve instead of sliding along it:
    25% general tokens preserved 98% of the domain gain while
    eliminating forgetting entirely.
  * Honest caveat: replay pool (WikiText train) shares the eval
    distribution (WikiText test). Splits are disjoint — no leakage —
    but the -17.9% partly reflects adaptation TOWARD the yardstick.
    Stricter design: replay from a third corpus.
  * Replay model learned WHEN to be Shakespeare, not to be Shakespeare
    unconditionally: modern prompts stayed modern, the king went full
    Elizabethan. Register became context-conditional.
  * Measurement discipline, again: identical harness before/after,
    pristine-checkpoint bit-exact verification (29.03/55.16), fresh
    optimizer+scaler per run, idempotent state-reset cells.

NEXT — NOTEBOOK 10 (Days 17-18): PARAMETER-EFFICIENT FINE-TUNING
  LoRA: freeze all 124M base weights, train low-rank adapters instead.
  The forgetting defense goes structural — the base model CANNOT drift
  because it never receives a gradient. Also: why r << d_model works
  (the low intrinsic dimension of the fine-tuning update), and QLoRA
  for 4-bit quantization on the T4.
""")
print("="*70)

NOTEBOOK 09 COMPLETE: FULL FINE-TUNING  (Days 15-16)

WHAT WE DID
  1. Loaded pretrained GPT-2 small (124,439,808 params) — our notebook-04
     architecture scaled up: same vocab (50257), same weight tying,
     learned pos-emb, pre-LN, GELU. d_model 768 / 12 layers / 12 heads.
  2. Baselines before touching weights: general PPL 29.03 (WikiText-2
     test, 286k tokens) + seeded generation snapshot on 3 fixed prompts.
  3. Domain corpus: Tiny Shakespeare, 338,025 BPE tokens (3.30 chars/tok
     vs ~4 for modern English — distribution shift visible at the
     tokenizer level). Contiguous 90/10 split. Domain baseline PPL 55.16.
  4. Naive fine-tune: 300 steps, batch 4 x seq 512, lr 3e-5 peak
     (warmup+cosine), fp16 + GradScaler, ~82 s on the T4.
  5. Replay fine-tune: identical run from pristine weights, but each
     batch = 3 Shakespeare + 1 WikiText-train sequence (25% replay).

THE 3-WAY VERDICT (full eval harness)
                          pristine   naive FT   replay FT
  gene